# Pandas `groupby` + `fillna` + `map` Practice

Practice notebook while learning Pandas.

Covers:
- Computing a per-group average with `groupby().mean()`
- Why `Series.fillna(other_series)` aligns by **index label**, not by category value
- Using `Series.map()` to translate a column of category labels into per-row values from a lookup Series
- Why `fillna()` (and most Pandas operations) return a **new** Series instead of modifying in place, and why the result has to be assigned back
- Correct order of operations: fill missing values *before* aggregating with `groupby().sum()`

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7],
    "category": ["cosmetics", "cosmetics", "electronics", "electronics",
                 "groceries", "groceries", "cosmetics"],
    "amount": [100, None, 500, 300, 50, 70, 300],
})

def summarize_sales(df):
    working_df = df.copy()

    avg_amount_per_category = working_df.groupby("category")["amount"].mean()

    # map() looks up each row's category in avg_amount_per_category (indexed by
    # category name) and returns a Series aligned to working_df's own row index (0..6).
    # A plain fillna(avg_amount_per_category) would NOT work here, because fillna
    # aligns by index label, and avg_amount_per_category is indexed by category
    # name (cosmetics/electronics/groceries), not by row number.
    row_level_avg = working_df["category"].map(avg_amount_per_category)

    # fillna() returns a new Series rather than modifying in place,
    # so the result must be assigned back into the column.
    working_df["amount"] = working_df["amount"].fillna(row_level_avg)

    # Totals computed AFTER the fill, so the missing value is included correctly.
    category_totals = working_df.groupby("category")["amount"].sum().sort_values(ascending=False)

    return working_df, avg_amount_per_category, category_totals

if __name__ == "__main__":
    working_df, avg_amount_per_category, category_totals = summarize_sales(orders)
    print("Filled data:")
    print(working_df)
    print("\nAverage amount per category:")
    print(avg_amount_per_category)
    print("\nCategory totals (descending):")
    print(category_totals)

### Expected output
```
Filled data:
   order_id     category  amount
0         1    cosmetics   100.0
1         2    cosmetics   200.0
2         3  electronics   500.0
3         4  electronics   300.0
4         5    groceries    50.0
5         6    groceries    70.0
6         7    cosmetics   300.0

Average amount per category:
category
cosmetics      200.0
electronics    400.0
groceries       60.0
Name: amount, dtype: float64

Category totals (descending):
category
cosmetics      600.0
electronics    800.0
groceries      120.0
Name: amount, dtype: float64
```

### Key takeaways
- `Series.fillna(other_series)` aligns by **index label**, not by matching column values. Passing a `groupby().mean()` result (indexed by category name) straight into `fillna()` on a Series indexed by row number silently fills nothing, because the labels never match.
- `Series.map(lookup)` walks a Series value-by-value, uses each value as a key into `lookup` (a dict or another Series), and returns a new Series **aligned to the original Series' own index** — this is what makes it possible to "broadcast" a per-category average back onto every row of that category.
- Most Pandas methods, including `fillna()`, return a new object rather than mutating in place. The result has to be explicitly reassigned (`working_df["amount"] = ...`) or the change is lost.
- Order of operations matters: aggregate (`groupby().sum()`) only *after* filling missing values, otherwise the aggregation runs on incomplete data.